# 05 · Reuse the decision patterns

**You will:** select a tool, filter a retrieved passage, check an output, and combine a decision with generation.

Use these small recipes as starting points. Thresholds below illustrate control flow; validate them on representative labelled data before adoption. Nothing here calls a real support tool, sends a message, or changes a customer account.

In [ ]:
# Find the course folder and load API settings without printing secret keys.
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'course_utils').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open Jupyter inside the course folder.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from course_utils.config import load_settings, require_key
settings = load_settings(ROOT)
print('Python:', sys.version.split()[0], '| Mode:', settings.mode)
print('Jev key configured:', bool(settings.typesafe_api_key))
print('OpenAI key configured:', bool(settings.openai_api_key))

## Setup: a small reusable request helper
Each example below defines its state and questions explicitly. This helper prints the input, calls Jev, and saves the response for replay. It does not run any support tools.

In [ ]:
# Import the decision types and configure live calls or authentic replay.
from contextlib import nullcontext
from typesafe_sdk import TypeSafeClient, Choice, Noul, RetryPolicy
from course_utils.records import run_case
from course_utils.evaluation import route_team

RUN_DIR = ROOT / 'work/runs'
can_jev = settings.mode == 'recorded' or bool(settings.typesafe_api_key)
can_openai = settings.mode == 'recorded' or bool(settings.openai_api_key)


def jev_client():
    if settings.mode == 'recorded':
        return nullcontext(None)
    return TypeSafeClient(
        api_key=require_key(settings, 'typesafe'),
        retry=RetryPolicy(max_retries=1),
        timeout=30,
    )

In [ ]:
# Send one decision request and show its input, status, and answers.
def ask_jev(example_id, state, questions):
    print('State analysed:')
    display(state)
    if not can_jev:
        print('Skipped: configure TYPESAFE_API_KEY or select authentic recordings.')
        return None

    config = {
        'state': state,
        'questions': {name: question.model_dump(mode='json') for name, question in questions.items()},
        'max_retries': 1,
        'timeout': 30,
    }
    with jev_client() as client:
        def request():
            return client.system_one(
                model=settings.jev_new, state=state, questions=questions,
            )

        result = run_case(
            request, provider='typesafe', endpoint='systemone',
            model=settings.jev_new, example_id=example_id, config=config,
            question_names=list(questions), mode=settings.mode, directory=RUN_DIR,
        )

    print('Status:', result.status, '| Error:', result.error)
    display(result.answers)
    return result

## 1. Select a tool
Ask which tool fits the request. This selects a name only; it does not invoke the tool or invent its arguments.

In [ ]:
# Select a read-only tool for an invoice question.
tool_state = {'request': 'Where can I find my invoice?'}
tool_questions = {
    'tool': Choice(
        instructions='Which read-only tool is relevant?',
        criteria={
            'invoice_lookup': 'Find existing invoices.',
            'status_lookup': 'Check service health.',
            'review': 'No clear match.',
        },
    ),
}
tool_result = ask_jev('tool_selection', tool_state, tool_questions)

## 2. Check a retrieved passage
Does the passage help answer the user’s question? Inspect the yes-probability before deciding whether to use it.

In [ ]:
# Check whether a retrieved passage helps answer a password-reset question.
relevance_state = {
    'question': 'How do I reset my password?',
    'passage': 'Open Account Settings, choose Security, then Reset password.',
}
relevance_questions = {
    'relevant': Noul(instructions='Does the passage directly help answer the question?'),
}
relevance_result = ask_jev('rag_relevance', relevance_state, relevance_questions)

## 3. Check a drafted reply
The draft claims a refund has happened. The policy only requires verification. Predict whether the claim is supported before running this cell. This model check is fallible, not a guarantee.

In [ ]:
# Check whether a drafted reply makes a claim unsupported by the supplied policy.
check_state = {
    'policy': 'Duplicate charges require verification before a refund.',
    'draft': 'Your refund has already been processed.',
}
check_questions = {
    'supported': Noul(instructions='Is every factual claim in the draft supported by the policy?'),
}
check_result = ask_jev('output_check', check_state, check_questions)

## 4. Route a ticket, then draft a reply - Cascading
First Jev chooses the team. Python sends uncertain answers to review. Only an accepted route proceeds to the optional OpenAI drafting step. The draft is displayed for human review, never sent.

In [ ]:
# Choose a support team for one ticket using a Choice question.
ticket_text = 'I was charged twice. Please refund the duplicate.'
route_state = {'ticket': ticket_text}
route_questions = {
    'team': Choice(
        instructions='Which team should handle the ticket? For mixed intents choose the first actionable issue.',
        criteria={
            'billing': 'Existing payments, refunds, invoices, and subscriptions.',
            'technical': 'Malfunctioning application features or service errors.',
            'account': 'Login, credentials, profiles, and membership.',
            'other': 'Sales, suggestions, general questions, or insufficient evidence.',
        },
    ),
}
route_result = ask_jev('hybrid-route', route_state, route_questions)

In [ ]:
# Apply an illustrative confidence threshold; missing answers go to human review.
destination = 'review'
if route_result is not None:
    team_answer = route_result.answers.get('team', {})
    destination = route_team(
        team_answer.get('choice', 'review'),
        team_answer.get('confidence', float('nan')),
        threshold=0.8,
    )
print('Destination:', destination)

### Prepare the writer
The schema asks for one string: `reply`. We preserve the raw response before validating the schema, so refusals and incomplete outputs remain visible. The helper records the request just like the Jev examples.

In [ ]:
# Define the reply schema and the rules the writer must follow.
from openai import OpenAI
from pydantic import BaseModel


class Reply(BaseModel):
    reply: str


reply_format = {
    'type': 'json_schema',
    'name': 'Reply',
    'strict': True,
    'schema': {**Reply.model_json_schema(), 'additionalProperties': False},
}
writer_instructions = (
    'Draft a short, considerate support reply. Do not claim actions have been taken. '
    'Policy: verify duplicate charges before any refund. '
    f'The team is {destination}.'
)

In [ ]:
# Call the LLM and record its response, including any refusal or incomplete output.
def draft_reply(client, ticket, instructions):
    def request():
        return client.responses.create(
            model=settings.openai_model,
            instructions=instructions,
            input=ticket,
            text={'format': reply_format},
            reasoning={'effort': 'none'},
            max_output_tokens=500,
            store=False,
        )

    config = {
        'instructions': instructions,
        'input': ticket,
        'schema': Reply.model_json_schema(),
        'reasoning': {'effort': 'none'},
        'max_output_tokens': 500,
        'store': False,
        'max_retries': 1,
        'timeout': 30,
    }
    return run_case(
        request, provider='openai', endpoint='responses',
        model=settings.openai_model, example_id='hybrid-draft', config=config,
        question_names=['reply'], mode=settings.mode, directory=RUN_DIR,
        response_schema=Reply,
    )

In [ ]:
# Generate a draft only when routing succeeds and the OpenAI step is available.
draft_record = None
if destination == 'review':
    print('Draft skipped: the ticket needs human review.')
elif not can_openai:
    print('Draft skipped: configure OPENAI_API_KEY or select authentic recordings.')
else:
    print('State analysed:')
    display({'ticket': ticket_text, 'instructions': writer_instructions})
    if settings.mode == 'recorded':
        writer = nullcontext(None)
    else:
        writer = OpenAI(api_key=require_key(settings, 'openai'), max_retries=1, timeout=30)

    with writer as client:
        draft_record = draft_reply(client, ticket_text, writer_instructions)

    print('Status:', draft_record.status, '| Error:', draft_record.error)
    print(draft_record.answers.get('reply', {}).get('value', 'No valid draft returned.'))

## Exercise

The output-check draft says a refund was processed, but the policy only authorizes verification. Predict the answer, inspect it, and rewrite the draft so that it makes no unsupported claim. Then decide what should happen when the relevance check is uncertain.

## Solution

A safer draft is: “Thanks for flagging the duplicate charge. Our billing team can check the transactions and confirm the next step.” For uncertain relevance, retain the passage for review or retrieve more evidence; do not treat a threshold as a universal truth. Expected semantic output-check answer is no, but inspect actual model errors rather than assuming it succeeds.

## Where to go next

| Resource | What to explore |
|---|---|
| [Official Python SDK](https://docs.typesafe.ai/sdk/python) | Client types, timeouts and retries |
| [Official SDK index](https://docs.typesafe.ai/sdk) | Other languages and integration entry points |
| [TypeSafe LLM adapter](https://github.com/typesafe-ai/system-one-adapter-python) | Using decision interfaces alongside LLM tooling; check current support |
| [Coding-agent guidance](https://docs.typesafe.ai/introduction/coding-agents) | Helping coding agents use the API |
| [Official patterns](https://docs.typesafe.ai/patterns) | Composition and review policies |
| [OpenAI Decisions](https://developers.openai.com/api/docs/guides/decisions) | Typed decisions within the OpenAI API |
| [Independent examples](https://systemonemodels.org/examples/) | Community experiments; assess evidence and maintenance yourself |

## Adoption checklist

Choose a concrete task and answer space. Label representative cases and failures. Measure error costs, review load, latency and total cost. Keep exact rules in code. Validate on held-out data, monitor drift, and rerun evaluation before upgrading models. Use generation when you need new text or a richer object; use ordinary code when you already know the rule.